# Notebook: Data Process

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter
import h5py
from sksurv.metrics import cumulative_dynamic_auc
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [ ]:
time_need = True
img_label = True

In [ ]:
def pandas_timestamp_to_pe(timestamp, d_model=1, ref_date=None):
    if ref_date is None:
        ref_date = pd.Timestamp('2000-01-01')
    

    days_since_ref = (timestamp - ref_date).days
    

    pos = days_since_ref
    pe = np.zeros(d_model)
    
    for i in range(0, d_model, 2):
        pe[i] = np.sin(pos / (10000 ** (i / d_model)))
        if i + 1 < d_model:
            pe[i + 1] = np.cos(pos / (10000 ** (i / d_model)))
    
    return pe

In [ ]:
def normalize_date(date_str):

    if pd.isna(date_str):
        return np.nan


    date_str = str(date_str).strip()


    date_str = date_str.replace('/', '-').replace('\\', '-')


    try:

        if date_str.isdigit() and len(date_str) == 8:
            return pd.to_datetime(date_str, format='%Y%m%d').strftime('%Y-%m-%d')


        elif date_str.isdigit() and len(date_str) == 6:
            return pd.to_datetime(date_str + '15', format='%Y%m%d').strftime('%Y-%m-%d')


        elif len(date_str) >= 8:

            try:
                dt = pd.to_datetime(date_str)
                return dt.strftime('%Y-%m-%d')
            except:

                parts = date_str.split('-')
                if len(parts) >= 2 and len(parts[0]) == 4:
                    year = parts[0]
                    month = parts[1].zfill(2)
                    return f"{year}-{month}-15"


        elif len(date_str) <= 7:
            parts = date_str.split('-')
            if len(parts) == 2:
                year = parts[0]
                month = parts[1].zfill(2)
                return f"{year}-{month}-15"

    except Exception as e:

        print(f"parse fail: {date_str}, error: {e}")
        return date_str

    return date_str

In [ ]:
baseline = pd.read_excel('baseline.xlsx')
dynamic = pd.read_excel('dynamic.xlsx')

In [ ]:
used_cols_dynamics = ["p_id","collection_date","NGLU","C-Pro","NCL","atypical_cells%"]
used_cols_baselines = ["id","性别","年龄","LM_DIAG_date","LMDFS","radical resection",
                       "smoking_status","primary_NGS_mutation","初诊cns转移","numberofBM","BM","os_date",
                       "os_status","脑放疗","line_therapy","LM_prog1_date","LM_prog2_date",
                       "LM_prog3_date","LM_prog4_date","LM_prog5_date"]

In [ ]:
dynamic = dynamic[used_cols_dynamics]
dynamic["id"] = dynamic["p_id"]
del dynamic["p_id"]

In [ ]:
baseline = baseline[used_cols_baselines]
baseline["gender"] = baseline["性别"]
baseline["age"] = baseline["年龄"]
baseline["CARE"] = baseline["初诊cns转移"]
baseline["Brain"] = baseline["脑放疗"]
del baseline["性别"], baseline["年龄"], baseline["脑放疗"], baseline["初诊cns转移"]

In [ ]:
tabular_input = pd.merge(left=baseline, right=dynamic, on="id", indicator=False,how="right")

In [ ]:
subset_cols = ['id', 'LM_DIAG_date', 'LMDFS', 'radical resection', 'smoking_status',
       'primary_NGS_mutation', 'numberofBM', 'BM', 'os_date', 'os_status',
       'LM_DIAG_date', 'line_therapy', 'gender', 'age',
       'CARE', 'Brain', 'collection_date', 'NGLU', 'C-Pro', 'NCL',"atypical_cells%"]

tabular_input.dropna(inplace=True, subset=subset_cols)
tabular_input["id"] = tabular_input["id"].astype(str)
tabular_input["id"] = tabular_input["id"].apply(lambda x: x.split(".")[0])

In [ ]:
def check_progression(row):
    lm_diag_date = row['collection_date']
    if pd.isna(lm_diag_date):
        return 0
    for i in range(1, 6):
        prog_col = f'LM_prog{i}_date'
        if prog_col not in row.index or pd.isna(row[prog_col]):
            continue
        prog_date = row[prog_col]
        if lm_diag_date < prog_date:
            lm_diag_plus_56 = lm_diag_date + pd.Timedelta(days=56)
            if lm_diag_plus_56 >= prog_date:
                return 1
            continue
    return 0

In [ ]:
tabular_input['LM_DIAG_date'] = pd.to_datetime(tabular_input['LM_DIAG_date'].apply(normalize_date))
tabular_input['collection_date'] = pd.to_datetime(tabular_input['collection_date'].apply(normalize_date))
tabular_input['os_date'] = pd.to_datetime(tabular_input['os_date'].apply(normalize_date))
tabular_input = tabular_input[tabular_input['os_date'].notna()]
tabular_input = tabular_input[tabular_input['collection_date'].notna()]
tabular_input = tabular_input[tabular_input['LM_DIAG_date'].notna()]
tabular_input["time_len"] = (tabular_input['collection_date'] - tabular_input['LM_DIAG_date']).dt.days
# non-float format feature processing
tabular_input = tabular_input[tabular_input["time_len"]>=0]
tabular_input['duration'] = (tabular_input['os_date'] - tabular_input['LM_DIAG_date']).dt.days
tabular_input = tabular_input[tabular_input["duration"]>=0]
tabular_input['LM_prog1_date'] = pd.to_datetime(tabular_input['LM_prog1_date'].apply(normalize_date))
tabular_input['LM_prog2_date'] = pd.to_datetime(tabular_input['LM_prog2_date'].apply(normalize_date))
tabular_input['LM_prog3_date'] = pd.to_datetime(tabular_input['LM_prog3_date'].apply(normalize_date))
tabular_input['LM_prog4_date'] = pd.to_datetime(tabular_input['LM_prog4_date'].apply(normalize_date))
tabular_input['LM_prog5_date'] = pd.to_datetime(tabular_input['LM_prog5_date'].apply(normalize_date))
# time processing
tabular_input['prog_risk'] = tabular_input.apply(check_progression, axis=1)

### Whether modal is ready

In [ ]:
modal_check = []

patient_id = tabular_input.id.tolist()
collection_date = tabular_input.collection_date.tolist()


with h5py.File("img_dataset.h5", 'r') as f:
    for idx in range(len(patient_id)):
        pt_id = str(patient_id[idx])
        c_date_raw = collection_date[idx]

        base_date = pd.to_datetime(c_date_raw).date()

        days_list = [(base_date + timedelta(days=i)).strftime('%Y%m%d') for i in range(-14, 1)]

        has_list = list(f[pt_id].keys()) if pt_id in f else []
        intersection = list(set(days_list) & set(has_list))
        
        found_valid_record = 0
        
        if intersection:
            for date_str in intersection:
                group = f[pt_id][date_str]
                keys_list = list(group.keys())

                required_keys = {'axi_img', 'cor_img', 'sag_img', 'mic_img'}
                if required_keys.issubset(set(keys_list)):

                    is_empty = False
                    for k in required_keys:
                        modality_group = group[k]
                        m_sum = 0
                        for slice_key in modality_group.keys():
                            m_sum += np.sum(modality_group[slice_key])
                        
                        if m_sum == 0:
                            is_empty = True
                            break
                    
                    if not is_empty:
                        found_valid_record = 1
                        break
        
        modal_check.append(found_valid_record)

tabular_input["modal_check"] = modal_check

### Past 84 days labels

In [ ]:
mri_has_label = []
mic_has_label = []

patient_id = tabular_input.id.tolist()
collection_date = tabular_input.collection_date.tolist()

with h5py.File("img_dataset.h5", 'a') as f:
    for idx in range(len(patient_id)):
        pt_id = str(patient_id[idx])
        c_date = str(pd.to_datetime(collection_date[idx])).split(" ")[0]
        base_date = datetime.strptime(c_date, '%Y-%m-%d').date()

        days_list = []
        for i in range(-84, 1):
            target_date = base_date + timedelta(days=i)
            days_list.append(target_date.strftime('%Y%m%d'))

        has_list = list(f[pt_id].keys()) if pt_id in f else []

        intersection = list(set(days_list) & set(has_list))

        mri_found = 0
        mic_found = 0

        if intersection:
            for date_str in intersection:
                keys_list = list(f[pt_id][date_str].keys())

                if not mri_found:
                    if any(img_type in keys_list for img_type in ['axi_img', 'cor_img', 'sag_img']):
                        mri_found = 1

                if not mic_found:
                    if 'mic_img' in keys_list:
                        mic_found = 1

                if mri_found and mic_found:
                    break

        mri_has_label.append(mri_found)
        mic_has_label.append(mic_found)


tabular_input["mri_84"] = mri_has_label
tabular_input["mic_84"] = mic_has_label

### Positional Encoding

In [ ]:
tabular_input["c_date_pos"] = tabular_input["collection_date"].apply(lambda x: float(pandas_timestamp_to_pe(x)[0]))

In [ ]:
split_date = pd.Timestamp('2024-07-31')
tabular_input['split_label'] = (tabular_input['LM_DIAG_date'] <= split_date).astype(int)

del tabular_input['LM_prog1_date'], tabular_input['LM_prog2_date'], tabular_input['LM_prog3_date'], tabular_input['LM_prog4_date'], tabular_input['LM_prog5_date']

In [ ]:
tabular_input.to_csv("tabular_input.csv", index=False, encoding='utf_8_sig')
# tabular_input_2 = tabular_input.copy()
# del tabular_input_2['c_date_pos'], tabular_input_2['time_len']
# tabular_input_2.to_csv("tabular_input_notime.csv", index=False, encoding='utf_8_sig')
# tabular_input_2 = tabular_input.copy()
# del tabular_input_2['mri_84'], tabular_input_2['mic_84']
# tabular_input_2.to_csv("tabular_input_noimglabel.csv", index=False, encoding='utf_8_sig')
# tabular_input_2 = tabular_input.copy()
# del tabular_input_2['c_date_pos'], tabular_input_2['time_len'], tabular_input_2['mri_84'], tabular_input_2['mic_84']
# tabular_input_2.to_csv("tabular_input_noimglabelnotime.csv", index=False, encoding='utf_8_sig')

In [ ]:
len(tabular_input[tabular_input["split_label"]==0])/len(tabular_input)